# Mapeamento do Universo Negociável de Ações na B3 — Granularidade Trimestral

Os arquivos de Séries Históricas da B3 contêm o registro diário consolidado de todas as negociações realizadas na bolsa, trazendo informações como tipo de ativo, data do pregão, código de negociação (ticker), modalidade de mercado, preços e volume financeiro. Esses arquivos não contêm exclusivamente ações de companhias abertas: junto com elas aparecem FIIs, ETFs, BDRs, derivativos, direitos de subscrição e diversos outros instrumentos financeiros que precisam ser eliminados para se chegar ao universo efetivamente investível.

Para isolar esse universo, a construção é feita em **três estágios progressivos e auditáveis**, cada um isolando um tipo de filtro e permitindo inspecionar visualmente o que sobrevive e o que é removido antes de avançar para o próximo:

1. **Estágio 1 — Registro, BDI e Mercado.** Seleciona apenas o mercado à vista e fracionário (BDIs 02 e 12, mercados 010 e 020), removendo metadados de arquivo, derivativos, opções e termo.
2. **Estágio 2 — Regex de ticker.** Restringe os tickers aos finais ordinários, preferenciais (incluindo classes adicionais PNC/PND) e units — 3, 4, 5, 6, 7, 8 e 11 — descartando BDRs (final 34/35/39 etc.), direitos e demais instrumentos com sufixos fora desse padrão.
3. **Estágio 3 — Remoção de fundos, BDRs e instrumentos de índice via `ESPECI`/`NOMRES`.** Elimina FIIs, FIPs, fundos em geral, BDRs (identificados pelo prefixo `DR` em `ESPECI`) e instrumentos atrelados a índices (ex: IBOV, IBrX), preservando ações cujo nome ou classificação contenha substrings potencialmente ambíguas (como "CI" ou "FIE" dentro do nome da empresa).

Após os três estágios, aplicamos um critério de **liquidez mínima por trimestre** — não mais por ano — para acompanhar a granularidade trimestral dos demonstrativos financeiros (ITR). Isso significa que um ticker pode ser considerado investível no Q1 de um ano e não no Q2 do mesmo ano, refletindo com mais fidelidade os períodos em que ele efetivamente tinha liquidez suficiente para compor uma simulação.

O corte de liquidez precisou ser recalibrado: os limiares originais (20 dias negociados / R$ 100.000,00 acumulados) foram definidos para uma janela de ~250 pregões por ano. Uma janela trimestral tem ~62 pregões, então aplicar o mesmo limiar absoluto penalizaria desproporcionalmente ativos legítimos apenas por estarem numa janela menor. Adotamos limiares proporcionais (não simplesmente 1/4): **10 dias negociados e R$ 30.000,00 de volume acumulado no trimestre**, mantendo uma margem de segurança superior ao corte linear (1/4 de 20 = 5 dias; 1/4 de R$100k = R$25k) para não relaxar demais o critério de "papel simulável". Esses valores são parâmetros e podem ser recalibrados conforme necessidade.

## Estrutura e Premissas do Layout B3

- Tipo de Registro (00-02) | Filtro: 01 — **Estágio 1**
    - O que é: Identifica o conteúdo da linha. O código 01 representa a cotação histórica diária.
    - Por que: Arquivos da B3 contêm metadados (00 e 99) e resumos estatísticos que não são preços. Usar apenas o 01 garante que você está lidando com dados brutos de negociação.

- Data da Transação (02-10) | Formato: AAAAMMDD — **usado no filtro de liquidez, pós Estágio 3**
    - O que é: O dia exato em que aqueles preços e volumes ocorreram.
    - Por que: Permite validar a frequência de negociação. Ações com presença em pregão inferior ao limiar estipulado (20 dias/ano) são descartadas para evitar papéis ilíquidos.

- Código BDI (10-12) | Filtro: 02 e 12 — **Estágio 1**
    - O que é: Classifica o tipo de papel (Ações, FIIs, BDRs, etc.). O código 02 representa o Lote Padrão e o 12 o Mercado Fracionário.
    - Por que: Isola o mercado acionário principal. Como FIIs, BDRs e ETFs também podem surgir nestes BDIs, este filtro é apenas o primeiro passo — atua em conjunto com o regex de ticker (Estágio 2) e as colunas de especificação/nome (Estágio 3) para garantir a captura exclusiva de ações de empresas.

- Ticker do Ativo (12-24) | Identificador — **Estágio 2**
    - O que é: O código da ação (ex: VALE3, PETR4). Ocupa 12 espaços para prever códigos longos, mas no Brasil usa 5 ou 6 caracteres seguidos de espaços.
    - Por que: É a chave primária para agregação dos dados e extração do radical da empresa (primeiros 4 caracteres). O regex `^[A-Z]{4}(3|4|5|6|7|8|11)$` restringe aos finais de ações e units, incluindo os finais 7 e 8 (classes adicionais de preferenciais, como PNC/PND — ex: `BRGE7`, `BRGE8`), evitando perder papéis legítimos por um padrão restritivo demais.

- Tipo de Mercado (24-27) | Filtro: 010 e 020 — **Estágio 1**
    - O que é: Indica a modalidade de negociação, sendo 010 o Mercado à Vista e 020 o Mercado Fracionário.
    - Por que: Remove operações com derivativos, opções e termo (070, 080), focando a análise no mercado acionário à vista.

- Nome Resumido e Especificação (27-49) | Filtro por Regex — **Estágio 3**
    - O que é: NOMRES (27–39) traz o nome da empresa e ESPECI (39–49) a classe do papel (ON, PN, DR, CI, etc.).
    - Por que usar: Filtro essencial para remover definitivamente FIIs, FIPs, fundos, BDRs (via prefixo `DR` em `ESPECI`) e instrumentos atrelados a índices (via prefixos como `IBO`/`IBX`/`SMA` em `ESPECI` ou pelo próprio ticker). Termos curtos como `CI` e `FIE` são aplicados com word boundary apenas sobre `ESPECI`, e não sobre `NOMRES`, para não remover indevidamente ações de empresas cujo nome contém essas substrings (ex: CIELO, CSN, DIRECIONAL, ESTÁCIO, CIA HERING, TRACK & FIELD, BROOKFIELD).

- Preço de Fechamento (108-121) | Cálculo: Valor / 100
    - O que é: O último preço negociado no dia. Vem sem vírgula (número inteiro com 2 casas decimais implícitas - ex: 1235 significa 12,35).
    - Por que: Serve como preço de referência para apuração de valor de mercado e cálculo de indicadores de valuation.

- Quantidade de Negócios (147-152) | Validação
    - O que é: Total de operações de compra e venda realizadas para aquele ativo no dia.
    - Por que: Permite atestar a distribuição da liquidez diária, evitando que um único grande negócio pontual mascare um ativo ilíquido.

- Volume Financeiro Total (170-188) | Cálculo: Valor / 100 — **usado no filtro de liquidez, pós Estágio 3**
    - O que é: Montante financeiro total em Reais negociado no dia (número inteiro com 2 casas decimais implícitas).
    - Por que: Utilizado no corte de liquidez mínima acumulada no ano (>= R$ 100.000,00), garantindo que a base final contenha apenas ativos com liquidez simulável.

# Construção do Universo Investível da B3

Os arquivos COTAHIST da B3 não contêm exclusivamente ações de companhias abertas, incluindo também ETFs, FIIs, FIAGROs, BDRs, derivativos, instrumentos atrelados a índices e diversos outros ativos financeiros. O objetivo deste notebook é construir, para cada ano, o conjunto de ações efetivamente negociáveis na B3, documentando cada etapa da seleção e justificando todos os filtros utilizados.

Antes de definir qualquer critério, realizamos uma auditoria exploratória do layout dos arquivos (valores únicos de `ESPECI` e `NOMRES`) para compreender quais categorias de ativos realmente aparecem na base histórica. Em seguida, os filtros são aplicados em três estágios isolados e auditáveis (Registro/BDI/Mercado → regex de ticker → remoção de fundos/BDR/índice), e só depois de validado o resultado do Estágio 3 é que aplicamos o corte de liquidez mínima na função final `processar_ano_apenas_acoes_empresas`.

In [1]:
import pandas as pd
import os
import json
from pathlib import Path

## Estrutura do arquivo COTAHIST

Os arquivos históricos da B3 possuem largura fixa (fixed width), em que cada posição representa um campo específico.  
Nesta etapa definimos apenas as colunas necessárias para construir o universo investível.

In [2]:
# Definições do Layout B3 (Largura Fixa)
COL_SPECS = [
    (0, 2),      # REG
    (2, 10),     # DATA
    (10, 12),    # BDI
    (12, 24),    # TICKER
    (24, 27),    # MERCADO
    (27, 39),    # NOMRES
    (39, 49),    # ESPECI
    (108, 121),  # PRECO
    (152, 170),  # QTD
    (170, 188),  # VOLUME
    (230, 242)   # ISIN
]

NAMES = [
    'REG',
    'DATA',
    'BDI',
    'TICKER',
    'MERCADO',
    'NOMRES',
    'ESPECI',
    'PRECO',
    'QTD',
    'VOLUME',
    'ISIN'
]


## Auditoria em Três Estágios dos Filtros de Universo Investível

Antes de consolidar a função final de construção do universo investível, vamos aplicar os três filtros de forma **isolada e auditável**, um de cada vez, sobre todos os anos disponíveis. A cada estágio, imprimimos o conjunto de tickers remanescentes (agregado de 2010 a 2025), para inspecionar visualmente:

1. **Estágio 1** — Filtro de Registro (REG=01), BDI (Lote Padrão/Fracionário) e Mercado (à vista/fracionário). Este é o filtro "óbvio e inegociável": remove metadados, derivativos, termo e opções.
2. **Estágio 2** — Filtro de regex no ticker (finais 3, 4, 5, 6, 7, 8 e 11). Os finais 7 e 8 foram incluídos após auditoria confirmar que representam classes adicionais de ações preferenciais (PNC/PND) de empresas reais (ex: `BRGE7`, `BRGE8`, `CELP7`, `EQPA7`, `TENE7`, `AXIA7`), não sendo exclusivos de BDRs ou fundos.
3. **Estágio 3** — Remoção de fundos (FII, FIP, FUNDO, FIE), BDRs (via prefixo `DR` em `ESPECI`) e instrumentos atrelados a índice (via `ESPECI`/ticker começando com `IBO`, `IBX`, `SMA`, `IDIV`). Termos curtos como `CI` e `FIE` são aplicados apenas sobre `ESPECI`, com word boundary, para não remover indevidamente ações cujo `NOMRES` contenha essas substrings por coincidência (ex: CIELO, CSN, DIRECIONAL, ESTÁCIO, TRACK & FIELD).

In [3]:
# --- ESTÁGIO 1: Filtro de Registro, BDI e Mercado ---

caminho_atual = Path.cwd()
pasta_anterior = caminho_atual.parent
diretorio = pasta_anterior / "data/raw/04_historical_series_b3_extracted"

tickers_estagio1 = set()
dados_brutos = {}

for arq in sorted(os.listdir(diretorio)):
    if not arq.endswith(".TXT"):
        continue

    caminho = os.path.join(diretorio, arq)
    ano_arquivo = "".join(filter(str.isdigit, arq))

    df = pd.read_fwf(caminho, colspecs=COL_SPECS, names=NAMES, skipfooter=1, encoding='latin1')

    # Filtro óbvio e inegociável: Registro de cotação diária + Mercado à vista/fracionário
    df = df[df['REG'] == 1].copy()
    df = df[df['BDI'].isin([2, 12])].copy()
    df = df[df['MERCADO'].isin([10, 20])].copy()

    df['TICKER'] = df['TICKER'].astype(str).str.strip()

    # --- Granularidade trimestral: deriva ano/mês/trimestre a partir da própria DATA do pregão,
    # não do nome do arquivo. Isso é mais robusto e já deixa os dados prontos para os estágios seguintes.
    df['DATA'] = df['DATA'].astype(str).str.zfill(8)
    df['ano'] = df['DATA'].str[:4].astype(int)
    df['mes_ref'] = df['DATA'].str[4:6].astype(int)
    df['trimestre'] = "Q" + (((df['mes_ref'] - 1) // 3) + 1).astype(str)
    df['periodo'] = df['ano'].astype(str) + df['trimestre']

    dados_brutos[ano_arquivo] = df
    tickers_estagio1.update(df['TICKER'].unique())

print("=" * 100)
print(f"ESTÁGIO 1 — Após filtro de Registro/BDI/Mercado")
print(f"Total de tickers distintos (todos os anos): {len(tickers_estagio1)}")
print("=" * 100)
print(sorted(tickers_estagio1))

ESTÁGIO 1 — Após filtro de Registro/BDI/Mercado
Total de tickers distintos (todos os anos): 2808
['A1AP34', 'A1BB34', 'A1BM34', 'A1CR34', 'A1DI34', 'A1DM34', 'A1EE34', 'A1EG34', 'A1EN34', 'A1EP34', 'A1ES34', 'A1FL34', 'A1GI34', 'A1GN34', 'A1IV34', 'A1JG34', 'A1KA34', 'A1LB34', 'A1LG34', 'A1LK34', 'A1LL34', 'A1LN34', 'A1LX34', 'A1MB34', 'A1MD34', 'A1ME34', 'A1MP34', 'A1MT34', 'A1MX34', 'A1NE34', 'A1NS34', 'A1NT34', 'A1ON34', 'A1OS34', 'A1PA34', 'A1PD34', 'A1PH34', 'A1RE34', 'A1RG34', 'A1SN34', 'A1SU34', 'A1TH34', 'A1TM34', 'A1TT34', 'A1UA34', 'A1UT34', 'A1VB34', 'A1VY34', 'A1WK34', 'A1YX34', 'A1ZN34', 'A2LC34', 'A2MB34', 'A2MC34', 'A2MR34', 'A2RE34', 'A2RR34', 'A2RW34', 'A2SO34', 'A2VL34', 'A2XO34', 'A2ZT34', 'AALC11B', 'AALC34', 'AALL34', 'AALR3', 'AAPL11B', 'AAPL34', 'ABBV34', 'ABCB4', 'ABCP11', 'ABCP12', 'ABEV3', 'ABGD39', 'ABNB3', 'ABRE11', 'ABRE3', 'ABTT11B', 'ABTT34', 'ABUD34', 'ABYA3', 'ACGU3', 'ACNB34', 'ADBE34', 'ADHM3', 'ADMF3', 'ADPR34', 'AEDU11', 'AEDU3', 'AEFI11', 'AELP3', 

In [4]:
# --- AUDITORIA GERAL: valores únicos de ESPECI e NOMRES na base bruta (pós Estágio 1) ---

especi_unicos = set()
nomres_unicos = set()

for ano, df in dados_brutos.items():
    especi_col = df['ESPECI'].dropna().astype(str).str.strip().str.upper()
    nomres_col = df['NOMRES'].dropna().astype(str).str.strip().str.upper()

    especi_unicos.update(especi_col.unique().tolist())
    nomres_unicos.update(nomres_col.unique().tolist())

# Garante que tudo é string antes de ordenar (proteção extra)
especi_unicos = {str(x) for x in especi_unicos}
nomres_unicos = {str(x) for x in nomres_unicos}

print("=" * 100)
print(f"VALORES ÚNICOS DE ESPECI NA BASE (todos os anos, pós Estágio 1)")
print(f"Quantidade: {len(especi_unicos)}")
print("=" * 100)
print(sorted(especi_unicos))

print("\n" + "=" * 100)
print(f"VALORES ÚNICOS DE NOMRES NA BASE (todos os anos, pós Estágio 1)")
print(f"Quantidade: {len(nomres_unicos)}")
print("=" * 100)
print(sorted(nomres_unicos))

VALORES ÚNICOS DE ESPECI NA BASE (todos os anos, pós Estágio 1)
Quantidade: 386
['CI', 'CI      MB', 'CI   EC', 'CI   ERC', 'CI  01', 'CI  ATZ', 'CI  ATZ MB', 'CI  ATZB', 'CI  EA', 'CI  EA  MB', 'CI  EB', 'CI  EB  MB', 'CI  EC', 'CI  ED', 'CI  EG', 'CI  EG  MB', 'CI  ER', 'CI  ER  MB', 'CI  ERA', 'CI  ERA MB', 'CI  ERB', 'CI  ERC', 'CI  ERG', 'CI  ERS', 'CI  ERS MB', 'CI  ES', 'CI  ES  MB', 'CI  ESA', 'CI  EX', 'CI  EX  MB', 'CI  I22', 'CI  P10', 'CI  P19', 'CI  S1', 'CI *', 'CI ER', 'CIA', 'CIA ER', 'DIR', 'DIR     MB', 'DIR  F3 MB', 'DIR  F4 MB', 'DIR CI', 'DIR CI  MB', 'DR1', 'DR1 ATZ', 'DR2', 'DR2 ATZ', 'DR2 EB', 'DR3', 'DR3 A', 'DR3 A EG', 'DR3 ATZ', 'DR3 B', 'DR3 B EG', 'DR3 EB', 'DR3 ED', 'DR3 EG', 'DR3 ES', 'DRE', 'DRE ED', 'DRN', 'DRN     MB', 'DRN A', 'DRN ATZ', 'DRN C', 'DRN EB', 'DRN EB  MB', 'DRN EBG', 'DRN EC', 'DRN EC  MB', 'DRN ECG', 'DRN ED', 'DRN ED  MB', 'DRN EDB', 'DRN EDC', 'DRN EDG', 'DRN EDR', 'DRN EG', 'DRN ER', 'DRN EX', 'IBO', 'IBO/', 'IBX', 'ON', 'ON      MA'

In [5]:
# --- ESTÁGIO 2: Filtro de regex no ticker ---

pattern = r'^[A-Z]{4}(3|4|5|6|7|8|11)$'

tickers_estagio2 = set()
dados_estagio2 = {}

for ano_arquivo, df in dados_brutos.items():
    df = df.copy()
    df['ISIN'] = df['ISIN'].astype(str).str.strip()
    df = df[df['TICKER'].str.match(pattern, na=False)].copy()

    dados_estagio2[ano_arquivo] = df
    tickers_estagio2.update(df['TICKER'].unique())

print("=" * 100)
print(f"ESTÁGIO 2 — Após filtro de regex de ticker (finais 3,4,5,6,7,8,11)")
print(f"Total de tickers distintos (todos os anos): {len(tickers_estagio2)}")
print("=" * 100)
print(sorted(tickers_estagio2))

# --- Referência: o que foi removido neste estágio (BDRs, ETFs internacionais, tickers de fração etc.) ---
perdidos = tickers_estagio1 - tickers_estagio2

print("\n" + "=" * 100)
print(f"TICKERS REMOVIDOS NO ESTÁGIO 2")
print(f"Quantidade total removida: {len(perdidos)}")
print("=" * 100)
print(sorted(perdidos))

ESTÁGIO 2 — Após filtro de regex de ticker (finais 3,4,5,6,7,8,11)
Total de tickers distintos (todos os anos): 1457
['AALR3', 'ABCB4', 'ABCP11', 'ABEV3', 'ABNB3', 'ABRE11', 'ABRE3', 'ABYA3', 'ACGU3', 'ADHM3', 'ADMF3', 'AEDU11', 'AEDU3', 'AEFI11', 'AELP3', 'AERI3', 'AESB3', 'AFCR11', 'AFHF11', 'AFHI11', 'AFLT3', 'AFLU3', 'AFLU5', 'AFOF11', 'AGCX11', 'AGEI3', 'AGEN11', 'AGIN3', 'AGRO3', 'AGXY3', 'AHEB3', 'AHEB5', 'AHEB6', 'AIEC11', 'AJFI11', 'ALLD3', 'ALLL11', 'ALLL3', 'ALLL4', 'ALMI11', 'ALOS3', 'ALPA3', 'ALPA4', 'ALPK3', 'ALSC3', 'ALSO3', 'ALUP11', 'ALUP3', 'ALUP4', 'ALZC11', 'ALZM11', 'ALZR11', 'ALZT11', 'AMAR3', 'AMBP3', 'AMBV3', 'AMBV4', 'AMCE3', 'AMER3', 'AMIL3', 'AMOB3', 'AMPI3', 'ANCR11', 'ANIM3', 'AORE3', 'APER3', 'APTI4', 'APTO11', 'APXM11', 'APXR11', 'APXU11', 'AQLL11', 'ARCT11', 'ARML3', 'ARND3', 'AROA11', 'ARRI11', 'ARTR3', 'ARXD11', 'ARZZ3', 'ASAI3', 'ASMT11', 'ASRF11', 'ATED3', 'ATOM3', 'ATSA11', 'AURB11', 'AURE3', 'AUTM3', 'AVIL3', 'AVLL3', 'AXIA3', 'AXIA5', 'AXIA6', 'AXI

In [6]:
# --- ESTÁGIO 3: Remoção de fundos via ESPECI/NOMRES ---

termo_fundo_generico = r'FII|FIP|FUNDO'
termo_fie_especi = r'(?:^|\s)FIE(?:\s|$)'
termo_ci_especi = r'(?:^|\s)CI(?:\s|$)'
termo_indice_especi = r'^IBO|^IBX|^SMA|^IDIV'
termo_bdr_especi = r'^DR'  # BDRs: DR1, DR2, DR3, DRE, DRN e variações

tickers_estagio3 = set()
dados_estagio3 = {}

for ano, df in dados_estagio2.items():
    df = df.copy()
    df['ESPECI'] = df['ESPECI'].astype(str).str.strip().str.upper()
    df['NOMRES'] = df['NOMRES'].astype(str).str.strip().str.upper()

    mask_fundos = (
        df['ESPECI'].str.contains(termo_fundo_generico, regex=True, na=False) |
        df['NOMRES'].str.contains(termo_fundo_generico, regex=True, na=False) |
        df['ESPECI'].str.contains(termo_fie_especi, regex=True, na=False) |
        df['ESPECI'].str.contains(termo_ci_especi, regex=True, na=False)
    )
    mask_indice = (
        df['TICKER'].str.match(r'^IBOV|^IBXL|^SMLL|^IDIV', na=False) |
        df['ESPECI'].str.contains(termo_indice_especi, regex=True, na=False) |
        df['NOMRES'].str.contains(r'IBOVESPA|SMALL CAP', regex=True, na=False)
    )
    mask_bdr = df['ESPECI'].str.contains(termo_bdr_especi, regex=True, na=False)

    df = df[~(mask_fundos | mask_indice | mask_bdr)].copy()

    dados_estagio3[ano] = df
    tickers_estagio3.update(df['TICKER'].unique())

print("=" * 100)
print(f"ESTÁGIO 3 — Após remoção de fundos (FII/FIP/FUNDO/CI/FIE)")
print(f"Total de tickers distintos (todos os anos): {len(tickers_estagio3)}")
print("=" * 100)
print(sorted(tickers_estagio3))

# O que foi removido neste estágio, para conferência
perdidos_fundos = sorted(tickers_estagio2 - tickers_estagio3)
print("\n" + "=" * 100)
print(f"TICKERS REMOVIDOS NO ESTÁGIO 3 (auditar se são realmente fundos)")
print(f"Quantidade: {len(perdidos_fundos)}")
print("=" * 100)
print(perdidos_fundos)

ESTÁGIO 3 — Após remoção de fundos (FII/FIP/FUNDO/CI/FIE)
Total de tickers distintos (todos os anos): 850
['AALR3', 'ABCB4', 'ABEV3', 'ABNB3', 'ABRE11', 'ABRE3', 'ABYA3', 'ACGU3', 'ADHM3', 'ADMF3', 'AEDU11', 'AEDU3', 'AELP3', 'AERI3', 'AESB3', 'AFLT3', 'AFLU3', 'AFLU5', 'AGEI3', 'AGIN3', 'AGRO3', 'AGXY3', 'AHEB3', 'AHEB5', 'AHEB6', 'ALLD3', 'ALLL11', 'ALLL3', 'ALLL4', 'ALOS3', 'ALPA3', 'ALPA4', 'ALPK3', 'ALSC3', 'ALSO3', 'ALUP11', 'ALUP3', 'ALUP4', 'AMAR3', 'AMBP3', 'AMBV3', 'AMBV4', 'AMCE3', 'AMER3', 'AMIL3', 'AMOB3', 'AMPI3', 'ANIM3', 'AORE3', 'APER3', 'APTI4', 'ARML3', 'ARND3', 'ARTR3', 'ARZZ3', 'ASAI3', 'ATED3', 'ATOM3', 'AURE3', 'AUTM3', 'AVIL3', 'AVLL3', 'AXIA3', 'AXIA5', 'AXIA6', 'AXIA7', 'AZEV3', 'AZEV4', 'AZTE3', 'AZUL4', 'AZZA3', 'BAHI3', 'BALM3', 'BALM4', 'BAUH4', 'BAZA3', 'BBAS3', 'BBDC3', 'BBDC4', 'BBRK3', 'BBSE3', 'BBTG11', 'BDLL3', 'BDLL4', 'BEEF3', 'BEES3', 'BEES4', 'BEMA3', 'BFRE11', 'BGIP3', 'BGIP4', 'BHGR3', 'BHIA3', 'BICB3', 'BICB4', 'BIDI11', 'BIDI3', 'BIDI4', 'BIE

## Consolidação: Da Auditoria em Estágios à Função Final

Com os três estágios validados individualmente — e a certeza de que os tickers remanescentes no Estágio 3 são ações reais, enquanto os removidos são de fato fundos, BDRs ou instrumentos de índice —, consolidamos a lógica em uma única função (`processar_ano_apenas_acoes_empresas`) que replica exatamente os mesmos três filtros e adiciona a etapa final: o **corte de liquidez mínima**.

Esse corte não é aplicado durante a auditoria dos estágios porque seu objetivo é diferente: os Estágios 1–3 filtram por *tipo de ativo* (é ação ou não?), enquanto o corte de liquidez filtra por *relevância prática* (mesmo sendo uma ação real, ela foi negociada o suficiente para ser simulável?). Um ativo pode sobreviver perfeitamente aos três estágios estruturais e ainda assim ser descartado aqui por ter menos de 20 dias negociados no ano ou menos de R$ 100.000,00 de volume financeiro acumulado — típico de ações que estrearam ou foram deslistadas no meio do ano, ou que simplesmente não têm liquidez para compor uma simulação realista.

In [7]:
LIMIAR_DIAS_TRIMESTRE = 10      # ajustável — ver justificativa no markdown acima
LIMIAR_VOLUME_TRIMESTRE = 30000  # ajustável — ver justificativa no markdown acima


def processar_arquivo_apenas_acoes_empresas(caminho_arquivo):
    """
    Processa um arquivo COTAHIST anual e retorna o universo investível
    em granularidade TRIMESTRAL: cada ticker aparece uma vez por trimestre
    em que atendeu aos critérios estruturais (Estágios 1-3) e de liquidez
    mínima (dias negociados e volume financeiro, ambos medidos DENTRO do
    trimestre, não do ano).
    """
    nome_arquivo = os.path.basename(caminho_arquivo)

    # 1. Leitura
    df = pd.read_fwf(caminho_arquivo, colspecs=COL_SPECS, names=NAMES, skipfooter=1, encoding='latin1')

    # 2. Filtro de Registro e Mercado
    df = df[df['REG'] == 1].copy()
    df = df[df['BDI'].isin([2, 12])].copy()
    df = df[df['MERCADO'].isin([10, 20])].copy()

    # 3. Derivação de período a partir da DATA do pregão
    df['DATA'] = df['DATA'].astype(str).str.zfill(8)
    df['ano'] = df['DATA'].str[:4].astype(int)
    df['mes_ref'] = df['DATA'].str[4:6].astype(int)
    df['trimestre'] = "Q" + (((df['mes_ref'] - 1) // 3) + 1).astype(str)
    df['periodo'] = df['ano'].astype(str) + df['trimestre']

    # 4. Limpeza de Ticker
    df['TICKER'] = df['TICKER'].str.strip()
    df['ISIN'] = df['ISIN'].astype(str).str.strip()
    pattern = r'^[A-Z]{4}(3|4|5|6|7|8|11)$'
    df = df[df['TICKER'].str.match(pattern, na=False)].copy()

    # --- FILTRO DEFINITIVO: REMOVER FIIS/FUNDOS E INSTRUMENTOS DE ÍNDICE ---
    df['ESPECI'] = df['ESPECI'].astype(str).str.strip().str.upper()
    df['NOMRES'] = df['NOMRES'].astype(str).str.strip().str.upper()

    termo_fundo_generico = r'FII|FIP|FUNDO'
    termo_fie_especi = r'(?:^|\s)FIE(?:\s|$)'
    termo_ci_especi = r'(?:^|\s)CI(?:\s|$)'
    termo_indice_especi = r'^IBO|^IBX|^SMA|^IDIV'
    termo_bdr_especi = r'^DR'

    mask_fundos = (
        df['ESPECI'].str.contains(termo_fundo_generico, regex=True, na=False) |
        df['NOMRES'].str.contains(termo_fundo_generico, regex=True, na=False) |
        df['ESPECI'].str.contains(termo_fie_especi, regex=True, na=False) |
        df['ESPECI'].str.contains(termo_ci_especi, regex=True, na=False)
    )
    mask_indice = (
        df['TICKER'].str.match(r'^IBOV|^IBXL|^SMLL|^IDIV', na=False) |
        df['ESPECI'].str.contains(termo_indice_especi, regex=True, na=False) |
        df['NOMRES'].str.contains(r'IBOVESPA|SMALL CAP', regex=True, na=False)
    )
    mask_bdr = df['ESPECI'].str.contains(termo_bdr_especi, regex=True, na=False)

    df = df[~(mask_fundos | mask_indice | mask_bdr)].copy()

    # 5. Ajuste de volume e liquidez mínima — agora agrupado por TICKER + PERIODO
    df['VOLUME'] = df['VOLUME'] / 100

    auditoria = (
        df.groupby(["TICKER", "periodo"], as_index=False)
        .agg(
            NOMRES=("NOMRES", "first"),
            ESPECI=("ESPECI", "first"),
            ISIN=("ISIN", "first"),
            ano=("ano", "first"),
            trimestre=("trimestre", "first"),
        )
    )

    stats = (
        df.groupby(["TICKER", "periodo"])
        .agg(
            dias_negociados=("DATA", "count"),
            volume_total=("VOLUME", "sum"),
            codigo_isin=("ISIN", "first"),
        )
        .reset_index()
    )

    stats = stats.merge(auditoria, on=["TICKER", "periodo"], how="left")

    # Filtro de liquidez mínima TRIMESTRAL
    acoes_ativas = stats[
        (stats['dias_negociados'] >= LIMIAR_DIAS_TRIMESTRE) &
        (stats['volume_total'] >= LIMIAR_VOLUME_TRIMESTRE)
    ].copy()

    acoes_ativas['RADICAL'] = acoes_ativas['TICKER'].str[:4]

    total_tickers = acoes_ativas['TICKER'].nunique()
    total_periodos = acoes_ativas['periodo'].nunique()
    total_empresas = acoes_ativas['RADICAL'].nunique()

    print(f"--- Processando: {nome_arquivo} ---")
    print(f"Tickers de AÇÕES REAIS (algum trimestre do arquivo): {total_tickers}")
    print(f"Trimestres distintos no arquivo: {total_periodos}")
    print(f"EMPRESAS distintas: {total_empresas}")
    print("-" * 40)

    return acoes_ativas

In [8]:
lista_frames = []

caminho_atual = Path.cwd()
pasta_anterior = caminho_atual.parent
diretorio = pasta_anterior / "data/raw/04_historical_series_b3_extracted"

for arq in os.listdir(diretorio):
    if arq.endswith(".TXT"):
        caminho = os.path.join(diretorio, arq)
        df_periodo = processar_arquivo_apenas_acoes_empresas(caminho)
        lista_frames.append(df_periodo)

df_final = pd.concat(lista_frames, ignore_index=True)
df_final = df_final.sort_values(by=['periodo', 'TICKER'], ascending=[True, True])

print("Total de linhas:", len(df_final))
print("Total de tickers:", df_final["TICKER"].nunique())
print("Total de períodos (trimestres):", df_final["periodo"].nunique())
print("Total de empresas:", df_final["RADICAL"].nunique())

--- Processando: COTAHIST_A2010.TXT ---
Tickers de AÇÕES REAIS (algum trimestre do arquivo): 438
Trimestres distintos no arquivo: 4
EMPRESAS distintas: 327
----------------------------------------
--- Processando: COTAHIST_A2011.TXT ---
Tickers de AÇÕES REAIS (algum trimestre do arquivo): 428
Trimestres distintos no arquivo: 4
EMPRESAS distintas: 326
----------------------------------------
--- Processando: COTAHIST_A2012.TXT ---
Tickers de AÇÕES REAIS (algum trimestre do arquivo): 399
Trimestres distintos no arquivo: 4
EMPRESAS distintas: 310
----------------------------------------
--- Processando: COTAHIST_A2013.TXT ---
Tickers de AÇÕES REAIS (algum trimestre do arquivo): 378
Trimestres distintos no arquivo: 4
EMPRESAS distintas: 300
----------------------------------------
--- Processando: COTAHIST_A2014.TXT ---
Tickers de AÇÕES REAIS (algum trimestre do arquivo): 368
Trimestres distintos no arquivo: 4
EMPRESAS distintas: 292
----------------------------------------
--- Processando

## Validação final dos ativos remanescentes

Com `df_final` já construído a partir da função consolidada — que aplica os três estágios de filtro estrutural (Registro/BDI/Mercado, regex de ticker com finais 3-8 e 11, remoção de fundos/BDR/índice) seguidos do corte de liquidez mínima —, realizamos uma auditoria de confirmação sobre os ativos que efetivamente compõem o universo investível final. O objetivo aqui não é mais descobrir novos padrões indevidos, mas validar que os filtros aplicados foram suficientes: a lista de "suspeitos" abaixo deve estar vazia ou conter apenas falsos positivos já conhecidos e aceitos (ex: `CCIM3` e `RDNI3` — nomes de empresas reais, "CC DES IMOB" e "RODOBENSIMOB", que contêm substrings coincidentes com termos de fundos imobiliários, mas cujo `ESPECI` confirma tratar-se de ações ordinárias comuns, `ON NM`).

In [9]:
print("=" * 100)
print("AUDITORIA DOS ATIVOS QUE SOBREVIVERAM AOS FILTROS")
print("=" * 100)

# Um registro por ticker
ativos = (
    df_final[
        ["TICKER", "NOMRES", "ESPECI", "codigo_isin"]
    ]
    .drop_duplicates()
    .sort_values("TICKER")
)

# Valores únicos de ESPECI
print("\nValores encontrados em ESPECI:\n")
print(
    ativos["ESPECI"]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 100)

# Valores únicos de NOMRES
print("\nNomes resumidos encontrados:\n")
print(
    ativos["NOMRES"]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 100)

# Possíveis ativos suspeitos remanescentes (checagem de confirmação)
padrao = (
    r"ETF|ETFS|FUNDO|INDICE|ÍNDICE|IBOV|SMAL|"
    r"BOVA|PIBB|HASH|TREND|XFIX|GOLD|"
    r"FII|FIP|FIAGRO|IMOB|BDR"
)
suspeitos = ativos[
    ativos["NOMRES"].str.contains(padrao, case=False, na=False)
    |
    ativos["ESPECI"].str.contains(padrao, case=False, na=False)
]
print(f"\nQuantidade de possíveis ativos não acionários: {len(suspeitos)}\n")
display(suspeitos.sort_values("TICKER"))

display(
    ativos
    .groupby("ESPECI")
    .agg(
        quantidade=("TICKER", "count"),
        exemplos=("TICKER", lambda x: ", ".join(sorted(x.unique())[:10]))
    )
    .sort_index()
)

AUDITORIA DOS ATIVOS QUE SOBREVIVERAM AOS FILTROS

Valores encontrados em ESPECI:

ESPECI
ON            174
ON      MA      5
ON      N1     39
ON      N2     31
ON      NM    324
             ... 
UNT EJ          2
UNT EJ  N2      8
UNT ES          1
UNT ES  N2      1
UNT EX  N2      1
Name: count, Length: 151, dtype: int64


Nomes resumidos encontrados:

NOMRES
3R PETROLEUM     1
3TENTOS          2
ABC BRASIL       3
ABNOTE           3
ABRIL EDUCA      4
                ..
WLM IND COM     10
YARA BRASIL      2
YBYRA S/A        1
YDUQS PART       2
ZAMP S.A.        2
Name: count, Length: 573, dtype: int64


Quantidade de possíveis ativos não acionários: 3



,TICKER,NOMRES,ESPECI,codigo_isin
290,CCIM3,CC DES IMOB,ON NM,BRCCIMACNOR5
1835,CCIM3,CC DES IMOB,ON ED NM,BRCCIMACNOR5
1178,RDNI3,RODOBENSIMOB,ON NM,BRRDNIACNOR9


,quantidade,exemplos
ESPECI,,
ON,174,"ABEV3, ADHM3, AELP3, AFLT3, AFLU3, AHEB3, AMBV..."
ON MA,5,"BAHI3, BIED3, BIOM3, NUTR3, SNSL3"
ON N1,39,"ALPA3, AXIA3, BBDC3, BICB3, BMEB3, BRAP3, BRKM..."
ON N2,31,"ABRE3, ALLL3, ALUP3, BIDI3, BPAC3, CLSC3, CMIN..."
ON NM,324,"AALR3, ABNB3, ABRE3, ABYA3, ACGU3, ADMF3, AEDU..."
...,...,...
UNT EJ,2,"BBTG11, SANB11"
UNT EJ N2,8,"BIDI11, CPLE11, KLBN11, SANB11, SAPR11, STBP11..."
UNT ES,1,ENGI11


## Construção da tabela de equivalência entre tickers e companhias

A B3 identifica ativos por ticker e ISIN, enquanto os demonstrativos financeiros da CVM utilizam o código CVM como identificador da companhia.  
Como não existe uma tabela pública consolidada relacionando diretamente esses identificadores ao longo de toda a série histórica, foi construída uma tabela de equivalência própria.  
A tabela contém, para cada ticker:

- ticker;
- código CVM;
- código ISIN;
- primeiro ano observado;
- último ano observado.

Os campos de código CVM são preenchidos manualmente apenas uma vez e preservados automaticamente em futuras atualizações da base.

In [10]:
pasta_interim = pasta_anterior / "data/interim"
pasta = pasta_interim / "05_investable_universe_of_stocks"
pasta.mkdir(parents=True, exist_ok=True)

# --- TABELA DE EQUIVALÊNCIA DOS TICKERS ---

caminho_mapping = pasta / "ticker_and_code_cvm_mapping.csv"
# Gera automaticamente a tabela a partir do universo investível
ticker_mapping_novo = (
    df_final
    .groupby("TICKER", as_index=False)
    .agg(
        codigo_isin=("codigo_isin", "first"),
        NOMRES=("NOMRES", "first"),
        primeiro_ano_observado=("ano", "min"),
        ultimo_ano_observado=("ano", "max")
    )
)
if not os.path.exists(caminho_mapping):
    ticker_mapping_novo.insert(1, "codigo_cvm", "")
    ticker_mapping_novo = ticker_mapping_novo[
        [
            "TICKER",
            "codigo_cvm",
            "codigo_isin",
            "NOMRES",
            "primeiro_ano_observado",
            "ultimo_ano_observado"
        ]
    ]
    ticker_mapping_novo.to_csv(
        caminho_mapping,
        sep=";",
        decimal=",",
        index=False,
        encoding="utf-8-sig"
    )
    print(f"\nTabela criada com {len(ticker_mapping_novo)} tickers.")
else:
    ticker_mapping_antigo = pd.read_csv(
        caminho_mapping,
        sep=";",
        dtype=str
    )
    ticker_mapping = ticker_mapping_novo.merge(
        ticker_mapping_antigo[["TICKER", "codigo_cvm"]],
        on="TICKER",
        how="left"
    )
    ticker_mapping = ticker_mapping[
        [
            "TICKER",
            "codigo_cvm",
            "codigo_isin",
            "NOMRES",
            "primeiro_ano_observado",
            "ultimo_ano_observado"
        ]
    ]
    ticker_mapping = ticker_mapping.sort_values("TICKER")
    ticker_mapping.to_csv(
        caminho_mapping,
        sep=";",
        decimal=",",
        index=False,
        encoding="utf-8-sig"
    )
    print(f"\nTabela atualizada com {len(ticker_mapping)} tickers.")

# --- ARQUIVO INTERMEDIÁRIO TICKER x PERÍODO (trimestral) ---
# Substitui o antigo ticker_per_year_raw.csv. Preserva ano, trimestre e periodo
# para que o notebook 05 (aplicação das exclusões de auditoria) consiga reconstruir
# o universo investível final na mesma granularidade trimestral.

ticker_periodo_raw = (
    df_final[["TICKER", "ano", "trimestre", "periodo"]]
    .drop_duplicates()
    .sort_values(["periodo", "TICKER"])
)
ticker_periodo_raw.to_csv(
    pasta / "ticker_per_period_raw.csv",
    sep=";", index=False
)
print(f"\nArquivo intermediário ticker×período salvo com {len(ticker_periodo_raw)} linhas.")


Tabela atualizada com 774 tickers.

Arquivo intermediário ticker×período salvo com 22937 linhas.


Após identificar quais tickers foram negociados na B3 em cada ano, o próximo passo consiste em estabelecer a correspondência entre cada ticker e a companhia emissora.

Embora a CVM disponibilize o código CVM, o CNPJ e a razão social das companhias abertas, os arquivos históricos da B3 (COTAHIST), utilizados neste trabalho para identificar os ativos negociados, contêm apenas informações como o ticker e o código ISIN, não existindo uma relação direta entre um elemento para identificar a companhia (código cvm ou cnpj) e seu ticker.

Além disso, essa correspondência não pode ser considerada estática ao longo do tempo. Durante o período analisado (2010-2025), diversas companhias passaram por eventos societários, como mudanças de ticker, incorporações, fusões, reorganizações societárias e alterações na estrutura do capital. Dessa forma, um mesmo ticker pode representar empresas diferentes em momentos distintos, assim como uma mesma companhia pode negociar sob tickers diferentes ao longo dos anos.

Por esse motivo, foi construída uma **tabela de equivalência (ticker_and_code_cvm_mapping.csv)** contendo, para cada ticker identificado nos arquivos históricos da B3, o respectivo código CVM e o intervalo de anos em que aquela associação é válida.

Os campos ticker, codigo ISIN, ano inicial e ano final são gerados automaticamente a partir dos arquivos históricos COTAHIST, percorrendo todos os anos disponíveis da base de dados.
O único campo preenchido manualmente é o código CVM. Esse preenchimento foi realizado por meio de consulta ao cadastro público de companhias abertas da CVM, disponível em:
https://cvmweb.cvm.gov.br/SWB/Sistemas/SCW/CPublica/CiaAb/FormBuscaCiaAb.aspx?TipoConsult=c

Essa etapa manual foi escolhida por oferecer maior confiabilidade do que a tentativa de estabelecer automaticamente a correspondência entre diferentes bases cadastrais da B3 e da CVM, além de envolver um número relativamente reduzido de registros (aproximadamente algumas centenas de tickers distintos), tornando o procedimento viável.

Ao incluir os campos ano_inicial e ano_final, a tabela preserva a validade temporal de cada correspondência, permitindo que os dados fundamentalistas de uma companhia sejam vinculados corretamente ao ticker utilizado em cada período do backtest e evitando associações incorretas decorrentes de mudanças cadastrais ao longo dos anos.